# Gold Processing — Order Items

Enriches `ecommerce.silver.silver_order_items` (183,378 rows) with four derived business columns into `ecommerce.gold.gold_order_items`:

* **`gross_amount`** = `unit_price * quantity`
* **`discount_amt`** = `gross_amount * discount_pct` (the discount fraction applied to gross; the amount actually discounted off the sale)
* **`sales_amount`** = `gross_amount - discount_amt + tax_amount` (net payable per line item)
* **`coupon_flag`** = `1` when `coupon_code` is present, `0` when null
* **`inr_rate`** — INR value of 1 unit of the row's currency. No rate file exists in the source data, so a static approximate mapping for the data period (Aug–Oct 2025) is used: INR 1.0, USD 86.0, GBP 115.0, AED 23.4, SGD 66.0, CAD 62.0, AUD 57.0
* **`sales_amt_INR`** = `sales_amount * inr_rate` — the sale converted to INR

All `silver_order_items` columns and metadata (`source_file`, `ingested_at`) pass through. Note: amounts are in mixed currencies — `unit_price_currency` stays alongside for any per-currency analysis.

In [0]:
from pyspark.sql.functions import col, when, lit, round as spark_round

target_table = "ecommerce.gold.gold_order_items"

df_gold_order_items = (
    spark.table("ecommerce.silver.silver_order_items")
    # gross_amount: unit price times quantity
    .withColumn("gross_amount", spark_round(col("unit_price") * col("quantity"), 2))
    # discount_amt: the discount fraction applied to gross
    .withColumn("discount_amt", spark_round(col("gross_amount") * col("discount_pct"), 2))
    # sales_amount: gross minus discount plus tax
    .withColumn(
        "sales_amount",
        spark_round(col("gross_amount") - col("discount_amt") + col("tax_amount"), 2),
    )
    # coupon_flag: 1 when a coupon code exists, else 0
    .withColumn("coupon_flag", when(col("coupon_code").isNotNull(), 1).otherwise(0))
    # inr_rate: INR value of 1 unit of the row's currency
    # (approximate static rates for the Aug-Oct 2025 data period — no rate file in the source)
    .withColumn(
        "inr_rate",
        when(col("unit_price_currency") == "INR", lit(1.0))
        .when(col("unit_price_currency") == "USD", lit(86.0))
        .when(col("unit_price_currency") == "GBP", lit(115.0))
        .when(col("unit_price_currency") == "AED", lit(23.4))
        .when(col("unit_price_currency") == "SGD", lit(66.0))
        .when(col("unit_price_currency") == "CAD", lit(62.0))
        .when(col("unit_price_currency") == "AUD", lit(57.0)),
    )
    # sales_amt_INR: sales amount converted to INR
    .withColumn("sales_amt_INR", spark_round(col("sales_amount") * col("inr_rate"), 2))
    .select(
        "dt",
        "order_ts",
        "customer_id",
        "order_id",
        "item_seq",
        "product_id",
        "quantity",
        "unit_price_currency",
        "unit_price",
        "discount_pct",
        "tax_amount",
        "gross_amount",
        "discount_amt",
        "sales_amount",
        "inr_rate",
        "sales_amt_INR",
        "channel",
        "coupon_code",
        "coupon_flag",
        "source_file",
        "ingested_at",
    )
)

# Idempotent full load into the gold table
# (overwriteSchema: the schema now includes inr_rate / sales_amt_INR — full rebuild replaces it)
(
    df_gold_order_items.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
# Verify the persisted gold table
spark.sql("""
    SELECT
      COUNT(*) AS rows,
      COUNT_IF(gross_amount != unit_price * quantity) AS bad_gross,
      COUNT_IF(discount_amt != round(gross_amount * discount_pct, 2)) AS bad_disc,
      COUNT_IF(sales_amount != round(gross_amount - discount_amt + tax_amount, 2)) AS bad_sales,
      COUNT_IF(coupon_flag != CASE WHEN coupon_code IS NOT NULL THEN 1 ELSE 0 END) AS bad_flag,
      COUNT_IF(inr_rate IS NULL) AS null_rates,
      COUNT_IF(sales_amt_INR != round(sales_amount * inr_rate, 2)) AS bad_inr_amt,
      SUM(coupon_flag) AS coupon_rows,
      ROUND(SUM(gross_amount), 2) AS total_gross,
      ROUND(SUM(sales_amount), 2) AS total_sales
    FROM ecommerce.gold.gold_order_items
""").show()

display(spark.table("ecommerce.gold.gold_order_items").limit(10))

# Per-currency INR conversion summary
spark.sql("""
    SELECT unit_price_currency, MIN(inr_rate) AS inr_rate, COUNT(*) AS rows,
           ROUND(SUM(sales_amt_INR), 2) AS total_sales_inr
    FROM ecommerce.gold.gold_order_items
    GROUP BY unit_price_currency
    ORDER BY total_sales_inr DESC
""").show()